In [0]:
# COMMAND ----------
# 1. Imports and configuration

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from functools import reduce

# Unity Catalog configuration
CATALOG = "capstone_8hospital"
SCHEMA = "hospital_schema"
VOLUME = "ext_capstone8_vol"

# Base Volume path
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Source data paths
PRIMARY_PATH = f"{BASE_PATH}/primary_events/"
STATUS_PATH = f"{BASE_PATH}/status_updates/"
SECONDARY_PATH = f"{BASE_PATH}/secondary_events/"

# Auto Loader schema and checkpoint paths
AUTOLOADER_SCHEMA_PATH = (f"{BASE_PATH}/autoloader_schema/primary_events")
PRIMARY_CHECKPOINT_PATH = (f"{BASE_PATH}/checkpoints/bronze_encounter_events")

# Bronze Delta tables
BRONZE_ENCOUNTER_TABLE = (f"{CATALOG}.{SCHEMA}.bronze_encounter_events")
BRONZE_CDC_TABLE = (f"{CATALOG}.{SCHEMA}.bronze_status_cdc")
BRONZE_SECONDARY_TABLE = (f"{CATALOG}.{SCHEMA}.bronze_secondary_events")

# Existing reference tables
BRONZE_DEPARTMENT_TABLE = (f"{CATALOG}.{SCHEMA}.department_bronze")
BRONZE_PATIENT_TABLE = (f"{CATALOG}.{SCHEMA}.patient_bronze")

# QA expectations from the supplied Healthcare Capstone 08 pack
EXPECTED_PRIMARY_ROWS = 11540
EXPECTED_UNIQUE_ENCOUNTERS = 11500
EXPECTED_DUPLICATE_ROWS = 40
EXPECTED_OPTIONAL_NULLS = 12
EXPECTED_OUTLIER_ROWS = 35
EXPECTED_CDC_ROWS = 520
EXPECTED_SECONDARY_ROWS = 310
EXPECTED_DEPARTMENT_ROWS = 110
EXPECTED_PATIENT_ROWS = 150

# Select catalog and schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Display configuration
print("Project: Healthcare Patient Journey and Hospital Operations")
print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Volume root:", BASE_PATH)


In [0]:
## 2. Verify the external volume folders and list source files
# MAGIC This is a read-only inspection of your uploaded source files. The notebook creates only the two
# MAGIC Auto Loader state directories if they do not already exist.

# COMMAND ----------
required_source_folders = {
    "primary_events": PRIMARY_PATH,
    "status_updates": STATUS_PATH,
    "secondary_events": SECONDARY_PATH,
}

for folder_name, folder_path in required_source_folders.items():
    try:
        entries = dbutils.fs.ls(folder_path)
        csv_files = [x for x in entries if x.name.lower().endswith(".csv")]
        print(f"{folder_name}: {len(csv_files)} CSV file(s)")
        for item in csv_files:
            print("   ", item.name, item.size, "bytes")
    except Exception as e:
        print(f"ERROR reading {folder_name} at {folder_path}: {e}")
        raise

# Ensure the Auto Loader state directories exist. Do not put source CSV files here.
dbutils.fs.mkdirs(AUTOLOADER_SCHEMA_PATH)
dbutils.fs.mkdirs(PRIMARY_CHECKPOINT_PATH)

In [0]:
## 3. Read all 12 primary CSV files for profiling
# MAGIC Each file is read separately and combined with `unionByName(allowMissingColumns=True)`.
# MAGIC This is deliberate: batches 10, 11 and 12 contain the additional `source_schema_version` column,
# MAGIC while earlier batches do not.

# COMMAND ----------
primary_file_infos = [
    x for x in dbutils.fs.ls(PRIMARY_PATH)
    if x.name.lower().endswith(".csv")
]
primary_file_infos = sorted(primary_file_infos, key=lambda x: x.name)

print("Primary CSV file count:", len(primary_file_infos))
display(spark.createDataFrame(
    [(x.name, x.path, x.size) for x in primary_file_infos],
    ["file_name", "file_path", "size_bytes"]
))

primary_dfs = []
primary_file_counts = []

for file_info in primary_file_infos:
    one_file_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(file_info.path)
        .withColumn("_source_file", F.lit(file_info.path))
    )
    primary_dfs.append(one_file_df)
    primary_file_counts.append(
        (file_info.name, one_file_df.count(), len(one_file_df.columns))
    )

if not primary_dfs:
    raise ValueError(f"No CSV files found in {PRIMARY_PATH}")

primary_raw = reduce(
    lambda left, right: left.unionByName(right, allowMissingColumns=True),
    primary_dfs
)

print("Primary total rows:", primary_raw.count())
print("Combined columns:", len(primary_raw.columns))
print("Combined column names:", primary_raw.columns)
display(primary_raw.limit(10))

In [0]:
## 4. File-level row counts and schema inspection

# COMMAND ----------
file_count_df = spark.createDataFrame(
    primary_file_counts,
    ["file_name", "row_count", "column_count"]
).orderBy("file_name")

display(file_count_df)

print("Files with 13 columns (schema-evolved batches):")
display(file_count_df.filter(F.col("column_count") == 13))

print("Primary schema:")
primary_raw.printSchema()

In [0]:
## 5. Validate schema evolution
# MAGIC
# MAGIC Expected: batches `flight_batch_10.csv`, `flight_batch_11.csv`, and `flight_batch_12.csv`
# MAGIC contain `source_schema_version`. Earlier batches should not contain that source column.
# MAGIC After union, the column exists in the combined DataFrame and is null for older batches.

# COMMAND ----------
schema_evolution_report = spark.createDataFrame(
    [
        (
            x.name,
            "source_schema_version" in next(
                d.columns for d, info in zip(primary_dfs, primary_file_infos)
                if info.name == x.name
            )
        )
        for x in primary_file_infos
    ],
    ["file_name", "has_source_schema_version"]
).orderBy("file_name")

display(schema_evolution_report)

if "source_schema_version" in primary_raw.columns:
    display(
        primary_raw
        .groupBy("_source_file", "source_schema_version")
        .count()
        .orderBy("_source_file", "source_schema_version")
    )
else:
    print("WARNING: source_schema_version was not detected in the combined source data.")


In [0]:
# COMMAND ----------
# 6. Profile primary row counts and duplicate encounter IDs

# Original healthcare business columns
base_columns = ["encounter_id","event_timestamp","ingest_timestamp","department_id","patient_id","visit_type",
    "encounter_status","priority","bill_amount","wait_minutes","doctor_id","source_system"]

# Validate expected columns
missing_base_columns = [c for c in base_columns if c not in primary_raw.columns]
if missing_base_columns:
    raise ValueError(
        f"Missing expected primary columns: {missing_base_columns}"
    )

# 1. Total source rows
primary_total_rows = primary_raw.count()

# 2. Distinct encounter IDs
primary_unique_encounter_ids = (
    primary_raw
    .select("encounter_id")
    .distinct()
    .count()
)

# 3. Encounter IDs appearing more than once
duplicate_encounter_id_groups = (
    primary_raw
    .groupBy("encounter_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("encounter_id")
)

# 4. Exact duplicate groups across the 12 original business columns
exact_duplicate_groups = (
    primary_raw
    .groupBy(*base_columns)
    .count()
    .filter(F.col("count") > 1)
)

# 5. Count extra rows caused by exact duplicates
duplicate_extra_rows = (
    exact_duplicate_groups
    .select(
        F.sum(F.col("count") - F.lit(1))
        .alias("duplicate_extra_rows")
    )
    .first()["duplicate_extra_rows"]
)

if duplicate_extra_rows is None:
    duplicate_extra_rows = 0

# Display results
print("Total primary rows:", primary_total_rows)
print("Distinct encounter IDs:",primary_unique_encounter_ids)
print("Encounter IDs occurring more than once:",duplicate_encounter_id_groups.count())
print("Exact duplicate groups:",exact_duplicate_groups.count())
print("Extra exact duplicate rows:",duplicate_extra_rows)

# Display duplicate records for inspection
display(duplicate_encounter_id_groups)
display(exact_duplicate_groups)


In [0]:
# COMMAND ----------
# 7. Profile null values

# Profile null values across all columns
null_profile = primary_raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in primary_raw.columns
])
display(null_profile)

# Check optional visit_type field
print(
    "visit_type null count:",
    primary_raw.filter(F.col("visit_type").isNull()).count()
)

# Check schema evolution metadata
if "source_schema_version" in primary_raw.columns:
    print(
        "source_schema_version null count (may be null for older batches):",
        primary_raw.filter(F.col("source_schema_version").isNull()).count()
    )


In [0]:

# COMMAND ----------
# 8. Profile encounter statuses

display(
    primary_raw
    .groupBy("encounter_status")
    .count()
    .orderBy("encounter_status")
)


In [0]:
# COMMAND ----------
# 9. Profile bill amounts, waiting minutes and potential outliers

primary_profile_typed = (
    primary_raw
    .withColumn("bill_amount_num",F.col("bill_amount").cast("double"))
    .withColumn("wait_minutes_num",F.col("wait_minutes").cast("double"))
)

# Summary statistics
display(
    primary_profile_typed
    .select(
        F.min("bill_amount_num").alias("min_bill_amount"),
        F.max("bill_amount_num").alias("max_bill_amount"),
        F.avg("bill_amount_num").alias("avg_bill_amount"),
        F.min("wait_minutes_num").alias("min_wait_minutes"),
        F.max("wait_minutes_num").alias("max_wait_minutes"),
        F.avg("wait_minutes_num").alias("avg_wait_minutes")
    )
)

outlier_bills = (
    primary_profile_typed
    .filter(F.col("bill_amount_num") > 5000)
    .select("encounter_id","bill_amount","wait_minutes","_source_file")
    .orderBy(F.col("bill_amount_num").desc())
)

print("Bill amounts greater than 5000:", outlier_bills.count())
display(outlier_bills)


In [0]:
# COMMAND ----------
# 10. Inspect encounter timestamps

display(
    primary_raw
    .select("event_timestamp", "ingest_timestamp")
    .orderBy("event_timestamp")
    .limit(10)
)

print(
    "Distinct event timestamps:",
    primary_raw.select("event_timestamp").distinct().count()
)

print(
    "Distinct ingestion timestamps:",
    primary_raw.select("ingest_timestamp").distinct().count()
)

In [0]:
## 11. Read and profile status CDC source

# COMMAND ----------
status_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")
    .load(STATUS_PATH)
)

print("CDC row count:", status_raw.count())
print("CDC columns:", status_raw.columns)
status_raw.printSchema()
display(status_raw.limit(10))

In [0]:
### CDC status distribution and sequence checks

# COMMAND ----------
display(
    status_raw
    .groupBy("new_status")
    .count()
    .orderBy("new_status")
)
# COMMAND ----------
# CDC status distribution and sequence checks

display(
    status_raw
    .groupBy("new_status")
    .count()
    .orderBy("new_status")
)

cdc_total = status_raw.count()

cdc_distinct_sequences = (status_raw.select("source_sequence").distinct().count())
cdc_duplicate_sequences = (status_raw.groupBy("source_sequence").count().filter(F.col("count") > 1))

print("CDC rows:", cdc_total)
print("Distinct source_sequence values:", cdc_distinct_sequences)
print("Duplicate source_sequence groups:", cdc_duplicate_sequences.count())

In [0]:
#12. Read and profile secondary events source

# COMMAND ----------
secondary_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")
    .load(SECONDARY_PATH)
)

print("Secondary event rows:", secondary_raw.count())
print("Secondary columns:", secondary_raw.columns)
secondary_raw.printSchema()
display(secondary_raw.limit(10))

In [0]:
# COMMAND ----------
# 13. Validate existing department and patient Delta tables

department_df = spark.table(BRONZE_DEPARTMENT_TABLE)
patient_df = spark.table(BRONZE_PATIENT_TABLE)

department_count = department_df.count()
patient_count = patient_df.count()

print("Department table:", BRONZE_DEPARTMENT_TABLE, "| rows:", department_count)
print("Patient table:", BRONZE_PATIENT_TABLE, "| rows:", patient_count)

print("Department schema:")
department_df.printSchema()

print("Patient schema:")
patient_df.printSchema()

display(department_df.limit(10))
display(patient_df.limit(10))

# COMMAND ----------
# Check uniqueness and nulls in reference keys

for table_name, df, key_col in [
    (BRONZE_DEPARTMENT_TABLE, department_df, "department_id"),
    (BRONZE_PATIENT_TABLE, patient_df, "patient_id")
]:
    print(f"\nValidation for {table_name}")
    print("Rows:", df.count())
    print("Distinct keys:", df.select(key_col).distinct().count())
    print("Null keys:", df.filter(F.col(key_col).isNull()).count())
    print(
        "Duplicate key groups:",
        df.groupBy(key_col)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

In [0]:
# COMMAND ----------
# 14. Source profiling reconciliation

def qa_status(actual, expected):
    return "PASS" if actual == expected else "CHECK"


observed_checks = [
    ("Primary rows", primary_total_rows, EXPECTED_PRIMARY_ROWS),
    ("Distinct encounter IDs", primary_unique_encounter_ids, EXPECTED_UNIQUE_ENCOUNTERS),
    ("Extra exact duplicate rows", int(duplicate_extra_rows or 0), EXPECTED_DUPLICATE_ROWS),
    (
        "Null visit_type rows",
        primary_raw.filter(F.col("visit_type").isNull()).count(),
        EXPECTED_OPTIONAL_NULLS
    ),
    ("CDC rows", cdc_total, EXPECTED_CDC_ROWS),
    ("Secondary event rows", secondary_raw.count(), EXPECTED_SECONDARY_ROWS),
    ("Department reference rows", department_count, EXPECTED_DEPARTMENT_ROWS),
    ("Patient reference rows", patient_count, EXPECTED_PATIENT_ROWS),
]

reconciliation_df = spark.createDataFrame(
    [
        (name, int(actual), int(expected), qa_status(actual, expected))
        for name, actual, expected in observed_checks
    ],
    ["check_name", "actual_value", "expected_value", "result"]
)

display(reconciliation_df)

In [0]:
# COMMAND ----------
# 15. Bronze ingestion — primary healthcare encounter feed with Auto Loader

primary_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", AUTOLOADER_SCHEMA_PATH)
    .option("rescuedDataColumn", "_rescued_data")
    .load(PRIMARY_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

primary_bronze_query = (
    primary_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PRIMARY_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_ENCOUNTER_TABLE)
)

primary_bronze_query.awaitTermination()

print("Auto Loader primary ingestion completed.")
print("Bronze table:", BRONZE_ENCOUNTER_TABLE)

# COMMAND ----------
# Validate primary Bronze table

bronze_encounters = spark.table(BRONZE_ENCOUNTER_TABLE)

bronze_primary_count = bronze_encounters.count()

bronze_distinct_encounters = (
    bronze_encounters
    .select("encounter_id")
    .distinct()
    .count()
)

print("Bronze encounter rows:", bronze_primary_count)
print("Distinct encounter IDs:", bronze_distinct_encounters)
print("Columns:", bronze_encounters.columns)

bronze_encounters.printSchema()

display(
    bronze_encounters
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

display(
    bronze_encounters
    .select("_source_file", "_ingested_at")
    .limit(10)
)

# COMMAND ----------
# Check Bronze duplicate retention and schema evolution

print(
    "Bronze rows beyond distinct encounter IDs:",
    bronze_primary_count - bronze_distinct_encounters
)

print(
    "source_schema_version present:",
    "source_schema_version" in bronze_encounters.columns
)

if "source_schema_version" in bronze_encounters.columns:
    display(
        bronze_encounters
        .groupBy("source_schema_version")
        .count()
        .orderBy("source_schema_version")
    )

In [0]:
# COMMAND ----------
# 16. Land CDC and secondary events in Bronze

def load_static_csv_to_bronze(source_path, target_table):
    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(source_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        source_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    loaded_df = spark.table(target_table)

    print(f"{target_table}: {loaded_df.count()} rows")

    return loaded_df


bronze_cdc = load_static_csv_to_bronze(
    STATUS_PATH,
    BRONZE_CDC_TABLE
)

bronze_secondary = load_static_csv_to_bronze(
    SECONDARY_PATH,
    BRONZE_SECONDARY_TABLE
)

In [0]:
# COMMAND ----------
# 17. Final Day 1 Bronze validation

bronze_validation_rows = [
    ("bronze_encounter_events", spark.table(BRONZE_ENCOUNTER_TABLE).count(), EXPECTED_PRIMARY_ROWS),
    ("bronze_status_cdc", spark.table(BRONZE_CDC_TABLE).count(), EXPECTED_CDC_ROWS),
    ("bronze_secondary_events", spark.table(BRONZE_SECONDARY_TABLE).count(), EXPECTED_SECONDARY_ROWS),
    ("department_bronze", spark.table(BRONZE_DEPARTMENT_TABLE).count(), EXPECTED_DEPARTMENT_ROWS),
    ("patients_bronze", spark.table(BRONZE_PATIENT_TABLE).count(), EXPECTED_PATIENT_ROWS),
]

bronze_validation_df = spark.createDataFrame(
    [
        (name, int(actual), int(expected), qa_status(actual, expected))
        for name, actual, expected in bronze_validation_rows
    ],
    ["table_name", "actual_rows", "expected_rows", "result"]
)

display(bronze_validation_df)

In [0]:
# COMMAND ----------
# Day 1 summary

day1_summary = [
    ("Primary profiling", primary_total_rows, EXPECTED_PRIMARY_ROWS),
    ("Primary Bronze", bronze_primary_count, EXPECTED_PRIMARY_ROWS),
    ("CDC Bronze", spark.table(BRONZE_CDC_TABLE).count(), EXPECTED_CDC_ROWS),
    ("Secondary Bronze", spark.table(BRONZE_SECONDARY_TABLE).count(), EXPECTED_SECONDARY_ROWS),
    ("Department reference", department_count, EXPECTED_DEPARTMENT_ROWS),
    ("Patient reference", patient_count, EXPECTED_PATIENT_ROWS),
]

display(
    spark.createDataFrame(
        [
            (name, int(actual), int(expected), qa_status(actual, expected))
            for name, actual, expected in day1_summary
        ],
        ["component", "actual_rows", "expected_rows", "result"]
    )
)